**Run after notebooks 10–11.** Derives all guide-quality tables/figures from `results/llm/judge_scores.csv` (single source → mutually consistent tables).

In [1]:
# 12_quality_analysis.ipynb
# Derives ALL guide-quality tables from results/llm/judge_scores.csv (single source -> consistent).
# Produces per-persona quality, primary-vs-independent agreement, by-sex means, DiCE-vs-SHAP t-tests,
# and a grayscale persona x dimension figure. RUN AFTER notebooks 10-11.

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LLM = ROOT / "results" / "llm"; TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

DIMS = ["D1_medical_accuracy", "D2_association_distinction", "D3_actionability",
        "D4_readability", "D5_disclaimer_adequacy", "D6_insurance_relevance", "D7_overall_quality"]
SHORT = {d: d.split("_")[0] for d in DIMS}
PERS = {"medical_expert": "Medical Expert", "insurance_specialist": "Insurance Specialist",
        "general_public": "General Public"}

df = pd.read_csv(LLM / "judge_scores.csv")
dice = df[df.method == "dice"]

# ---- (i) DiCE quality by dimension x persona (primary judge) ----
prim = dice[dice.judge == "primary"]
rows = []
for d in DIMS:
    row = {"Dimension": SHORT[d]}
    for p, pl in PERS.items():
        s = prim[prim.persona == p][d]
        row[pl] = f"{s.mean():.2f} +/- {s.std():.2f}"
    rows.append(row)
qual = pd.DataFrame(rows)
qual.loc[len(qual)] = {"Dimension": "Mean(D1-D7)",
                       **{pl: f"{prim[prim.persona==p][DIMS].mean().mean():.2f}" for p, pl in PERS.items()}}
qual.to_csv(TAB / "table_quality_by_persona.csv", index=False)
print("DiCE guide quality by dimension x persona (primary judge):")
print(qual.to_string(index=False))

# ---- (ii) Cross-model judge agreement (primary vs independent), per dimension ----
agg = (dice.groupby(["case_id", "persona", "judge"])[DIMS].mean().reset_index())
rows = []
for d in DIMS:
    piv = agg.pivot_table(index=["case_id", "persona"], columns="judge", values=d)
    pm, im = piv["primary"], piv["independent"]
    if pm.std() == 0 or im.std() == 0:
        r, lab = np.nan, ("Constant" if pm.std() == 0 else "---")
    else:
        r = stats.pearsonr(pm, im)[0]
        lab = "Strong" if r >= 0.8 else "Moderate" if r >= 0.5 else "Weak"
    rows.append([SHORT[d], f"{pm.mean():.2f} +/- {pm.std():.2f}", f"{im.mean():.2f} +/- {im.std():.2f}",
                 round(r, 3) if not np.isnan(r) else "n/a", round(abs(pm.mean()-im.mean()), 3), lab])
cross = pd.DataFrame(rows, columns=["Dim", "Primary", "Independent", "Pearson_r", "Mean_abs_diff", "Agreement"])
cross.to_csv(TAB / "table_cross_model_agreement.csv", index=False)
print("\nCross-model judge agreement (primary vs independent; both OpenAI-family):")
print(cross.to_string(index=False))

# ---- (iii) By-sex mean scores (all personas, primary judge) ----
rows = []
for d in DIMS:
    row = {"Dimension": SHORT[d]}
    for sx in ["Male", "Female"]:
        row[sx] = round(prim[prim.sex == sx][d].mean(), 2)
    rows.append(row)
gender = pd.DataFrame(rows)
gender.to_csv(TAB / "table_gender_scores.csv", index=False)
print("\nBy-sex mean scores (consistent with the tables above by construction):")
print(gender.to_string(index=False))

# ---- (iv) DiCE vs SHAP: actionability (D3) & readability (D4) paired t-tests ----
rows = []
for d, lab in [("D3_actionability", "D3 Actionability"), ("D4_readability", "D4 Readability")]:
    for p, pl in PERS.items():
        a = df[(df.method == "dice") & (df.judge == "primary") & (df.persona == p)].set_index("case_id")[d]
        b = df[(df.method == "shap") & (df.judge == "primary") & (df.persona == p)].set_index("case_id")[d]
        common = a.index.intersection(b.index)
        if len(common) > 1 and (a[common] - b[common]).std() > 0:
            t, pv = stats.ttest_rel(a[common], b[common])
        else:
            t, pv = np.nan, np.nan
        rows.append([lab, pl, round(a.mean(), 2), round(b.mean(), 2),
                     round(t, 2) if not np.isnan(t) else "n/a", f"{pv:.3g}" if not np.isnan(pv) else "n/a"])
cmp = pd.DataFrame(rows, columns=["Dimension", "Persona", "DiCE_mean", "SHAP_mean", "t", "p"])
cmp.to_csv(TAB / "table_dice_vs_shap_quality.csv", index=False)
print("\nDiCE vs SHAP guide quality (paired t-tests, primary judge):")
print(cmp.to_string(index=False))

# ---- Figure: persona x dimension (DiCE, primary judge), grayscale ----
mat = np.array([[prim[prim.persona == p][d].mean() for d in DIMS] for p in PERS])
fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(mat, cmap="Greys", vmin=1, vmax=5, aspect="auto")
ax.set_xticks(range(len(DIMS))); ax.set_xticklabels([SHORT[d] for d in DIMS])
ax.set_yticks(range(len(PERS))); ax.set_yticklabels(list(PERS.values()))
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.1f}", ha="center", va="center",
                color="white" if mat[i, j] > 3 else "black", fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="Mean score (1-5)")
savefig(fig, "fig_llm_quality")
print("\nSaved quality tables and fig_llm_quality.")

DiCE guide quality by dimension x persona (primary judge):
  Dimension Medical Expert Insurance Specialist General Public
         D1  3.10 +/- 1.17        3.20 +/- 0.97  3.23 +/- 1.03
         D2  4.62 +/- 0.54        4.65 +/- 0.48  4.67 +/- 0.62
         D3  1.88 +/- 0.76        1.93 +/- 0.73  1.98 +/- 0.70
         D4  3.98 +/- 0.36        4.05 +/- 0.32  3.95 +/- 0.38
         D5  4.45 +/- 0.50        4.47 +/- 0.51  4.67 +/- 0.47
         D6  3.52 +/- 0.64        4.05 +/- 0.22  3.25 +/- 0.59
         D7  3.27 +/- 0.91        3.50 +/- 0.60  3.40 +/- 0.78
Mean(D1-D7)           3.55                 3.69           3.59

Cross-model judge agreement (primary vs independent; both OpenAI-family):
Dim       Primary   Independent Pearson_r  Mean_abs_diff Agreement
 D1 3.17 +/- 1.05 2.92 +/- 1.18     0.809          0.250    Strong
 D2 4.65 +/- 0.54 4.90 +/- 0.30     0.502          0.250  Moderate
 D3 1.93 +/- 0.72 1.69 +/- 0.56     0.769          0.233  Moderate
 D4 4.00 +/- 0.35 4.01 +/- 0.46